
# アイパターンによるシグナルインテグリティ評価（シミュレーション）

このノートブックは、アイパターン（アイダイアグラム）を用いたシグナルインテグリティ（SI）評価の基本を、簡易なチャネル・ジッタ・雑音モデルで再現します。  
参照: onsemi Application Note AND9075 "Understanding Data Eye Diagram Methodology for Analyzing High Speed Digital Signals"。

本ノートブックで行うこと
- NRZ/PRBS 信号の生成
- ISI と帯域制限を含むチャネルモデル
- ランダムジッタ（RJ）とデターミニスティックジッタ（DJ）の付加
- アイパターン描画
- 眼開口・眼幅・クロッシング・ジッタの簡易推定



## 評価指標の要点（概要）

- アイダイアグラムは、波形を 1 UI（Unit Interval）ごとに折り畳んで重ね合わせた統計的表示。
- 眼開口（Eye Height）は縦方向の開きで、雑音による閉じを示す。
- 眼幅（Eye Width）はクロッシング点の統計的平均間隔で、水平タイミングマージンの指標。
- Eye Crossing % はクロッシングレベルを 1/0 レベルで正規化した指標。
- ジッタは理想タイミングからの時間偏差で、ランダム成分と決定論的成分がある。


In [ ]:

import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({
    "figure.figsize": (10, 4),
    "axes.grid": True,
})


In [ ]:

params = {
    "data_rate_gbps": 5.0,
    "samples_per_ui": 64,
    "n_bits": 4000,
    "amplitude": 1.0,
    "use_prbs": True,
    "prbs_order": 7,
    "seed": 7,
    "isi_taps": [1.0, 0.30, -0.12],
    "rc_alpha": 0.08,
    "rc_bw_ratio": 0.60,
    "rc_poles": 2,
    "auto_timing": True,
    "auto_timing_mode": "xcorr",
    "timing_offset_ui": 0.0,
    "rj_sigma_ui": 0.006,
    "dj_amp_ui": 0.010,
    "dj_freq_ui": 0.020,
    "noise_sigma": 0.030,
    "eye_traces": 800,
}
params


In [ ]:

def prbs7(n_bits, seed=0x5A):
    state = seed & 0x7F
    if state == 0:
        state = 0x5A
    bits = np.empty(n_bits, dtype=np.int8)
    for i in range(n_bits):
        new_bit = ((state >> 6) ^ (state >> 5)) & 1
        bits[i] = state & 1
        state = ((state << 1) | new_bit) & 0x7F
    return bits


def generate_bits(n_bits, use_prbs=True, order=7, seed=1):
    if use_prbs:
        if order != 7:
            raise ValueError("This demo supports PRBS7 only.")
        return prbs7(n_bits, seed=seed)
    rng = np.random.default_rng(seed)
    return rng.integers(0, 2, size=n_bits, dtype=np.int8)


def nrz_waveform(bits, samples_per_ui, amplitude=1.0):
    symbols = (2 * bits - 1) * amplitude
    return np.repeat(symbols.astype(float), samples_per_ui)


In [ ]:

def apply_isi(x, taps):
    h = np.array(taps, dtype=float)
    return np.convolve(x, h, mode="same")


def isi_taps_from_ui(gains, samples_per_ui):
    """Build sample-spaced FIR taps from UI-spaced gains.

    gains[0] -> 0 UI (main cursor)
    gains[1] -> +1 UI, gains[2] -> +2 UI, ...
    """
    gains = np.asarray(gains, dtype=float)
    if gains.size == 0:
        return np.array([1.0], dtype=float)
    taps = np.zeros((len(gains) - 1) * samples_per_ui + 1, dtype=float)
    for i, g in enumerate(gains):
        taps[i * samples_per_ui] = g
    return taps


def lowpass_1pole(x, alpha):
    y = np.empty_like(x, dtype=float)
    y[0] = x[0]
    for i in range(1, len(x)):
        y[i] = y[i - 1] + alpha * (x[i] - y[i - 1])
    return y


def rc_alpha_from_bw_ratio(bw_ratio, samples_per_ui):
    """Convert 3dB bandwidth ratio (f3dB/data_rate) to alpha."""
    return 1.0 - np.exp(-2.0 * np.pi * bw_ratio / samples_per_ui)


def lowpass_rc(x, alpha, poles=1):
    y = x
    for _ in range(poles):
        y = lowpass_1pole(y, alpha)
    return y


def apply_jitter(x, samples_per_ui, rj_sigma_ui=0.0, dj_amp_ui=0.0, dj_freq_ui=0.0, seed=0):
    rng = np.random.default_rng(seed)
    t = np.arange(len(x)) / samples_per_ui
    rj = rng.normal(0.0, rj_sigma_ui, size=len(x))
    dj = dj_amp_ui * np.sin(2 * np.pi * dj_freq_ui * t)
    t_query = t + rj + dj
    y = np.interp(t_query, t, x, left=x[0], right=x[-1])
    return y


def add_noise(x, sigma=0.0, seed=0):
    rng = np.random.default_rng(seed)
    return x + rng.normal(0.0, sigma, size=len(x))


def simulate_signal(p):
    bits = generate_bits(
        p["n_bits"],
        use_prbs=p["use_prbs"],
        order=p["prbs_order"],
        seed=p["seed"],
    )
    x = nrz_waveform(bits, p["samples_per_ui"], amplitude=p["amplitude"])
    y = apply_isi(x, p["isi_taps"])
    alpha = p["rc_alpha"]
    if p.get("rc_bw_ratio") is not None:
        alpha = rc_alpha_from_bw_ratio(p["rc_bw_ratio"], p["samples_per_ui"])
    y = lowpass_rc(y, alpha, p.get("rc_poles", 1))
    y = apply_jitter(
        y,
        p["samples_per_ui"],
        rj_sigma_ui=p["rj_sigma_ui"],
        dj_amp_ui=p["dj_amp_ui"],
        dj_freq_ui=p["dj_freq_ui"],
        seed=p["seed"] + 1,
    )
    y = add_noise(y, sigma=p["noise_sigma"], seed=p["seed"] + 2)
    return bits, y


In [ ]:

def eye_segments_centered(y, samples_per_ui, ui_span=2, center_offset=0):
    seg_len = int(ui_span * samples_per_ui)
    half = seg_len // 2
    centers = np.arange(samples_per_ui // 2 + center_offset, len(y), samples_per_ui)
    centers = centers[(centers >= half) & (centers + half < len(y))]
    segments = np.stack([y[c - half : c + half] for c in centers])
    t = (np.arange(seg_len) - half) / samples_per_ui
    return segments, t


def estimate_timing_offset(y, bits, samples_per_ui):
    best_phase = 0
    best_height = -np.inf
    for phase in range(samples_per_ui):
        center_samples = y[phase::samples_per_ui]
        bit_center = bits[: len(center_samples)]
        one_center = center_samples[bit_center == 1]
        zero_center = center_samples[bit_center == 0]
        if one_center.size < 10 or zero_center.size < 10:
            continue
        height = float(np.percentile(one_center, 5) - np.percentile(zero_center, 95))
        if height > best_height:
            best_height = height
            best_phase = phase
    return int(best_phase - samples_per_ui // 2)


def estimate_timing_offset_crossing(y, bits, samples_per_ui):
    # Estimate center offset from median crossing time.
    segments, t_eye = eye_segments_centered(y, samples_per_ui, ui_span=2, center_offset=0)
    center_idx = int(0.5 * samples_per_ui)
    center_samples = y[center_idx::samples_per_ui]
    bit_center = bits[: len(center_samples)]
    one_center = center_samples[bit_center == 1]
    zero_center = center_samples[bit_center == 0]
    if one_center.size < 10 or zero_center.size < 10:
        return 0
    threshold = float((np.mean(one_center) + np.mean(zero_center)) / 2)
    left, right = crossing_times(segments, t_eye, threshold)
    crossings = []
    if left.size:
        crossings.append(left)
    if right.size:
        crossings.append(right)
    if not crossings:
        return 0
    mean_cross = float(np.median(np.concatenate(crossings)))
    return int(round(mean_cross * samples_per_ui))


def estimate_timing_offset_xcorr(y, bits, samples_per_ui, max_ui=2000):
    # Cross-correlation between received waveform and ideal NRZ to estimate delay.
    n_ui = min(len(bits), int(max_ui))
    if n_ui < 10:
        return 0
    x = nrz_waveform(bits[:n_ui], samples_per_ui, amplitude=1.0)
    y_seg = y[: len(x)].astype(float)
    x = x - np.mean(x)
    y_seg = y_seg - np.mean(y_seg)
    n = len(x)
    nfft = 1
    while nfft < 2 * n - 1:
        nfft *= 2
    X = np.fft.rfft(x, n=nfft)
    Y = np.fft.rfft(y_seg, n=nfft)
    corr = np.fft.irfft(Y * np.conj(X), n=nfft)
    corr = corr[: 2 * n - 1]
    lag = int(np.argmax(corr) - (n - 1))
    # Convert lag to phase offset within a UI, centered around 0.
    offset = lag % samples_per_ui
    if offset > samples_per_ui // 2:
        offset -= samples_per_ui
    return int(offset)


def crossing_times(segments, t, threshold):
    left = []
    right = []
    for seg in segments:
        s = seg - threshold
        idx = np.where(np.diff(np.signbit(s)))[0]
        if idx.size == 0:
            continue
        for i in idx:
            t0, t1 = t[i], t[i + 1]
            y0, y1 = s[i], s[i + 1]
            if y1 == y0:
                continue
            tc = t0 - y0 * (t1 - t0) / (y1 - y0)
            if tc < 0:
                left.append(tc)
            else:
                right.append(tc)
    return np.array(left), np.array(right)


def eye_metrics(y, bits, samples_per_ui, center_offset=0):
    half_window = int(0.1 * samples_per_ui)
    center_idx = int(0.5 * samples_per_ui) + center_offset
    start = center_idx - half_window
    end = center_idx + half_window
    one_samples = []
    zero_samples = []
    for i in range(len(bits)):
        base = i * samples_per_ui
        if base + end <= len(y) and base + start >= 0:
            window = y[base + start : base + end]
            if bits[i] == 1:
                one_samples.append(window)
            else:
                zero_samples.append(window)
    one_samples = np.concatenate(one_samples) if one_samples else np.array([0.0])
    zero_samples = np.concatenate(zero_samples) if zero_samples else np.array([0.0])

    one_level = float(np.mean(one_samples))
    zero_level = float(np.mean(zero_samples))
    eye_amplitude = one_level - zero_level

    centers = np.arange(center_idx, len(y), samples_per_ui)
    centers = centers[centers >= 0]
    center_samples = y[centers]
    bit_center = bits[: len(center_samples)]
    one_center = center_samples[bit_center == 1]
    zero_center = center_samples[bit_center == 0]
    eye_height = float(np.percentile(one_center, 5) - np.percentile(zero_center, 95))

    segments, t_eye = eye_segments_centered(y, samples_per_ui, ui_span=2, center_offset=center_offset)
    cross_win = 0.02
    cross_mask = np.abs(t_eye) <= (cross_win / 2)
    crossing_level = float(np.mean(segments[:, cross_mask]))
    eye_cross_pct = 100.0 * (crossing_level - zero_level) / eye_amplitude

    threshold = (one_level + zero_level) / 2
    left, right = crossing_times(segments, t_eye, threshold)
    mean_left = float(np.mean(left)) if left.size else float("nan")
    mean_right = float(np.mean(right)) if right.size else float("nan")
    eye_width_ui = float(mean_right - mean_left) if left.size and right.size else float("nan")

    jitter_dev = np.concatenate([left + 0.5, right - 0.5]) if left.size and right.size else np.array([0.0])
    jitter_pp_ui = float(np.ptp(jitter_dev))
    jitter_rms_ui = float(np.std(jitter_dev))

    return {
        "one_level": one_level,
        "zero_level": zero_level,
        "eye_amplitude": eye_amplitude,
        "eye_height": eye_height,
        "eye_cross_pct": eye_cross_pct,
        "eye_width_ui": eye_width_ui,
        "jitter_pp_ui": jitter_pp_ui,
        "jitter_rms_ui": jitter_rms_ui,
    }


In [ ]:

bits, y = simulate_signal(params)

center_offset = 0
if params.get("auto_timing", False):
    mode = params.get("auto_timing_mode", "eye_height")
    if mode == "xcorr":
        center_offset = estimate_timing_offset_xcorr(y, bits, params["samples_per_ui"])
    elif mode == "crossing":
        center_offset = estimate_timing_offset_crossing(y, bits, params["samples_per_ui"])
    else:
        center_offset = estimate_timing_offset(y, bits, params["samples_per_ui"])
else:
    center_offset = int(round(params.get("timing_offset_ui", 0.0) * params["samples_per_ui"]))

metrics = eye_metrics(y, bits, params["samples_per_ui"], center_offset=center_offset)

ui_s = 1.0 / (params["data_rate_gbps"] * 1e9)

def ui_to_ps(x):
    return x * ui_s * 1e12

print("Eye Height         : {:.4f} V".format(metrics["eye_height"]))
print("Eye Width          : {:.4f} UI ({:.2f} ps)".format(metrics["eye_width_ui"], ui_to_ps(metrics["eye_width_ui"])))
print("Eye Crossing %     : {:.2f} %".format(metrics["eye_cross_pct"]))
print("Jitter p-p         : {:.4f} UI ({:.2f} ps)".format(metrics["jitter_pp_ui"], ui_to_ps(metrics["jitter_pp_ui"])))
print("Jitter RMS         : {:.4f} UI ({:.2f} ps)".format(metrics["jitter_rms_ui"], ui_to_ps(metrics["jitter_rms_ui"])))


In [ ]:

t = np.arange(len(y)) / params["samples_per_ui"]
view_ui = 8
n = view_ui * params["samples_per_ui"]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(t[:n], y[:n], lw=1)
axes[0].set_xlabel("Time [UI]")
axes[0].set_ylabel("Amplitude")
axes[0].set_title("Time Waveform (first 8 UI)")

segments, t_eye = eye_segments_centered(y, params["samples_per_ui"], ui_span=2, center_offset=center_offset)
trace_n = min(params["eye_traces"], len(segments))
for seg in segments[:trace_n]:
    axes[1].plot(t_eye, seg, color="C0", alpha=0.05)

th = (metrics["one_level"] + metrics["zero_level"]) / 2
axes[1].axhline(th, color="k", ls="--", lw=1)
axes[1].axvline(0, color="k", ls=":", lw=1)
axes[1].set_xlabel("Time relative to bit center [UI]")
axes[1].set_ylabel("Amplitude")
axes[1].set_title("Eye Diagram")

plt.tight_layout()



## パラメータの調整

- `rj_sigma_ui` を増やすと水平開口が閉じ、ジッタが増える。
- `noise_sigma` を増やすと縦方向の開口が閉じる。
- `isi_taps` や `rc_alpha` を変えると立上り・立下りが遅くなり、アイが閉じる。


In [ ]:
# --- Parameter Scan (widgets + batch sweep) ---
try:
    import ipywidgets as widgets
    from IPython.display import display
    HAVE_WIDGETS = True
except Exception:
    HAVE_WIDGETS = False


def run_with_params(rc_bw_ratio=None, rc_poles=None, noise_sigma=None, rj_sigma_ui=None, dj_amp_ui=None):
    p = dict(params)
    if rc_bw_ratio is not None:
        p["rc_bw_ratio"] = float(rc_bw_ratio)
    if rc_poles is not None:
        p["rc_poles"] = int(rc_poles)
    if noise_sigma is not None:
        p["noise_sigma"] = float(noise_sigma)
    if rj_sigma_ui is not None:
        p["rj_sigma_ui"] = float(rj_sigma_ui)
    if dj_amp_ui is not None:
        p["dj_amp_ui"] = float(dj_amp_ui)

    bits, y = simulate_signal(p)
    center_offset = 0
    if p.get("auto_timing", False):
        mode = p.get("auto_timing_mode", "eye_height")
        if mode == "xcorr":
            center_offset = estimate_timing_offset_xcorr(y, bits, p["samples_per_ui"])
        elif mode == "crossing":
            center_offset = estimate_timing_offset_crossing(y, bits, p["samples_per_ui"])
        else:
            center_offset = estimate_timing_offset(y, bits, p["samples_per_ui"])
    else:
        center_offset = int(round(p.get("timing_offset_ui", 0.0) * p["samples_per_ui"]))

    metrics = eye_metrics(y, bits, p["samples_per_ui"], center_offset=center_offset)

    t = np.arange(len(y)) / p["samples_per_ui"]
    view_ui = 8
    n = view_ui * p["samples_per_ui"]

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(t[:n], y[:n], lw=1)
    axes[0].set_xlabel("Time [UI]")
    axes[0].set_ylabel("Amplitude")
    axes[0].set_title("Time Waveform (first 8 UI)")

    segments, t_eye = eye_segments_centered(y, p["samples_per_ui"], ui_span=2, center_offset=center_offset)
    trace_n = min(p["eye_traces"], len(segments))
    for seg in segments[:trace_n]:
        axes[1].plot(t_eye, seg, color="C0", alpha=0.05)

    th = (metrics["one_level"] + metrics["zero_level"]) / 2
    axes[1].axhline(th, color="k", ls="--", lw=1)
    axes[1].axvline(0, color="k", ls=":", lw=1)
    axes[1].set_xlabel("Time relative to bit center [UI]")
    axes[1].set_ylabel("Amplitude")
    axes[1].set_title("Eye Diagram")
    plt.tight_layout()


def scan_param_grid(rc_bw_list, rc_poles_list, noise_list):
    rc_bw_list = list(rc_bw_list)
    rc_poles_list = list(rc_poles_list)
    noise_list = list(noise_list)

    fig, axes = plt.subplots(1, len(rc_poles_list), figsize=(4 * len(rc_poles_list), 3), squeeze=False)
    best = {"eye_height": -1e9}

    for j, poles in enumerate(rc_poles_list):
        eye_h = np.zeros((len(noise_list), len(rc_bw_list)), dtype=float)
        for i, noise in enumerate(noise_list):
            for k, bw in enumerate(rc_bw_list):
                p = dict(params)
                p["rc_bw_ratio"] = float(bw)
                p["rc_poles"] = int(poles)
                p["noise_sigma"] = float(noise)

                bits, y = simulate_signal(p)
                center_offset = 0
                if p.get("auto_timing", False):
                    mode = p.get("auto_timing_mode", "eye_height")
                    if mode == "xcorr":
                        center_offset = estimate_timing_offset_xcorr(y, bits, p["samples_per_ui"])
                    elif mode == "crossing":
                        center_offset = estimate_timing_offset_crossing(y, bits, p["samples_per_ui"])
                    else:
                        center_offset = estimate_timing_offset(y, bits, p["samples_per_ui"])
                else:
                    center_offset = int(round(p.get("timing_offset_ui", 0.0) * p["samples_per_ui"]))

                metrics = eye_metrics(y, bits, p["samples_per_ui"], center_offset=center_offset)
                eye_h[i, k] = metrics["eye_height"]
                if metrics["eye_height"] > best["eye_height"]:
                    best = {
                        "eye_height": metrics["eye_height"],
                        "rc_bw_ratio": bw,
                        "rc_poles": poles,
                        "noise_sigma": noise,
                    }

        ax = axes[0, j]
        im = ax.imshow(
            eye_h,
            origin="lower",
            aspect="auto",
            extent=[min(rc_bw_list), max(rc_bw_list), min(noise_list), max(noise_list)],
        )
        ax.set_title(f"rc_poles={poles}")
        ax.set_xlabel("rc_bw_ratio")
        ax.set_ylabel("noise_sigma")
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    plt.tight_layout()
    print("Best:", best)


if HAVE_WIDGETS:
    w = widgets.interact(
        run_with_params,
        rc_bw_ratio=widgets.FloatSlider(min=0.2, max=1.0, step=0.05, value=params["rc_bw_ratio"], description="rc_bw"),
        rc_poles=widgets.IntSlider(min=1, max=6, step=1, value=params["rc_poles"], description="poles"),
        noise_sigma=widgets.FloatSlider(min=0.0, max=0.08, step=0.005, value=params["noise_sigma"], description="noise"),
        rj_sigma_ui=widgets.FloatSlider(min=0.0, max=0.02, step=0.001, value=params["rj_sigma_ui"], description="rj"),
        dj_amp_ui=widgets.FloatSlider(min=0.0, max=0.05, step=0.002, value=params["dj_amp_ui"], description="dj"),
    )
    display(w)
else:
    print("ipywidgets not available. Use scan_param_grid(...) for batch scan.")
